# Muhammad Umar Iftikhar
# News Topic Classifier — Fine-Tuned BERT on AG News

**DevelopersHub Corporation — Internship Task 1**

End-to-end pipeline: load AG News → fine-tune `bert-base-uncased` → evaluate → deploy a live Streamlit UI via a public tunnel from Colab.

> **Runtime:** Enable a GPU in Colab (*Runtime → Change runtime type → T4 GPU*) before running training cells.

## Cell 1: Environment Setup & Installations

Install Hugging Face ecosystem packages, evaluation utilities, and deployment tools (`streamlit`, tunnel helpers).

In [10]:
# Cell 1 — Environment setup & quiet installs
import subprocess
import sys
from typing import List


def install_packages(packages: List[str]) -> None:
    """Install pip packages with minimal console noise."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *packages],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )


# Do NOT pip-install torch on Colab — it often breaks the torch/torchvision pair
# and causes: RuntimeError: operator torchvision::nms does not exist
REQUIRED_PACKAGES = [
    "transformers",
    "datasets",
    "accelerate",
    "scikit-learn",
    "streamlit",
    "pyngrok",
]

try:
    install_packages(REQUIRED_PACKAGES)
    print("All dependencies installed successfully.")
except subprocess.CalledProcessError as exc:
    raise RuntimeError(f"Package installation failed: {exc.stderr.decode()}") from exc

# If Cell 3 still fails with a torchvision::nms error, run this once, then Runtime → Restart session:
# !pip install -q --upgrade torch torchvision torchaudio

All dependencies installed successfully.


## Cell 2: Dataset Loading & Tokenization

Load the [AG News](https://huggingface.co/datasets/ag_news) dataset and tokenize headlines + descriptions with `bert-base-uncased` (max length 128).

In [11]:
# Cell 2 — Dataset loading & tokenization
from typing import Any, Dict

import torch
from datasets import DatasetDict, load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

# AG News label index → human-readable topic (Hugging Face uses 0–3)
ID_TO_LABEL: Dict[int, str] = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
NUM_LABELS = len(ID_TO_LABEL)


def load_ag_news() -> DatasetDict:
    """Load the AG News dataset from Hugging Face Hub."""
    try:
        dataset = load_dataset("ag_news")
    except Exception as exc:
        raise RuntimeError(f"Failed to load ag_news: {exc}") from exc
    return dataset


def tokenize_batch(
    examples: Dict[str, Any],
    tokenizer: AutoTokenizer,
    max_length: int = MAX_LENGTH,
) -> Dict[str, Any]:
    """
    Tokenize a batch of AG News examples.

    Each example has a single 'text' field (title + description).
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )


raw_datasets = load_ag_news()
print(raw_datasets)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized_datasets = raw_datasets.map(
    lambda batch: tokenize_batch(batch, tokenizer),
    batched=True,
    desc="Tokenizing ag_news",
)

# Trainer expects torch tensors; drop raw text column
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

print(f"Train samples: {len(train_dataset):,} | Test samples: {len(test_dataset):,}")
print(f"Sample tokenized keys: {train_dataset[0].keys()}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
Train samples: 120,000 | Test samples: 7,600
Sample tokenized keys: dict_keys(['label', 'input_ids', 'attention_mask'])


## Cell 3: Model Setup & Training Arguments

Load `AutoModelForSequenceClassification`, configure Colab-friendly `TrainingArguments`, and define evaluation metrics (Accuracy + Weighted F1).

In [13]:
# Cell 3 — Model, training arguments, and metrics
from typing import Dict

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
)

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
print(f"Using device: {DEVICE} (fp16={USE_CUDA})")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={str(k): v for k, v in ID_TO_LABEL.items()},
    label2id={v: str(k) for k, v in ID_TO_LABEL.items()},
)

training_args = TrainingArguments(
    output_dir="./bert_ag_news_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=USE_CUDA,
    dataloader_num_workers=2,
    seed=42,
)

def compute_metrics(eval_pred) -> Dict[str, float]:
    """Compute Accuracy and Weighted F1 for the Trainer (scikit-learn)."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "f1": float(f1_score(labels, predictions, average="weighted")),
    }

print("Cell 3 complete — ready for training:")
print(f"  model: {type(model).__name__}")
print(f"  training_args: {training_args.output_dir}")
print(f"  train_dataset: {len(train_dataset):,} samples")

Using device: cuda (fp16=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Cell 3 complete — ready for training:
  model: BertForSequenceClassification
  training_args: ./bert_ag_news_checkpoints
  train_dataset: 120,000 samples


## Cell 4: Model Fine-Tuning & Evaluation

Train with the Hugging Face `Trainer`, evaluate on the test split, print a classification report, and persist artifacts to `./saved_bert_model/`.

In [14]:
# Cell 4 — Fine-tuning, evaluation, and model persistence
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report
from transformers import Trainer

# Variables created in Cells 2–3 must exist in this kernel session
_REQUIRED_VARS = [
    "model",
    "training_args",
    "compute_metrics",
    "train_dataset",
    "test_dataset",
    "tokenizer",
    "ID_TO_LABEL",
    "NUM_LABELS",
]
_missing = [name for name in _REQUIRED_VARS if name not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing: {_missing}. Run Cell 2, then Cell 3, then Cell 4 "
        "(same session — do not restart runtime between them)."
    )

SAVE_DIR = Path("./saved_bert_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

_trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# transformers >= 4.46 uses processing_class; older versions use tokenizer
try:
    trainer = Trainer(**_trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Trainer(**_trainer_kwargs, tokenizer=tokenizer)

print("Starting fine-tuning…")
train_result = trainer.train()
print(f"Training complete. Global step: {train_result.global_step}")

print("\nEvaluating on the test set…")
eval_metrics = trainer.evaluate(eval_dataset=test_dataset)
print("\n=== Final Test Metrics ===")
for metric_name, value in sorted(eval_metrics.items()):
    if isinstance(value, float):
        print(f"  {metric_name}: {value:.4f}")
    else:
        print(f"  {metric_name}: {value}")

# Detailed per-class report
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\n=== Classification Report ===")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        digits=4,
    )
)

trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"\nModel and tokenizer saved to: {SAVE_DIR.resolve()}")

Starting fine-tuning…


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.190589,0.185681,0.943421,0.943420
2,0.121932,0.182519,0.948158,0.948188
3,0.085164,0.217271,0.949079,0.949117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.085164,0.217271,3,0.949079,0.949117



=== Final Test Metrics ===
  eval_accuracy: 0.9491
  eval_f1: 0.9491
  eval_loss: 0.2173



=== Classification Report ===
              precision    recall  f1-score   support

       World     0.9670    0.9574    0.9622      1900
      Sports     0.9895    0.9884    0.9889      1900
    Business     0.9305    0.9095    0.9199      1900
    Sci/Tech     0.9104    0.9411    0.9255      1900

    accuracy                         0.9491      7600
   macro avg     0.9494    0.9491    0.9491      7600
weighted avg     0.9494    0.9491    0.9491      7600



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and tokenizer saved to: /content/saved_bert_model


## Cell 5: Writing the Streamlit Application File

The cell below writes `app.py` to disk. It loads the saved checkpoint, accepts news text, and displays the predicted topic with confidence.

In [15]:
%%writefile app.py
"""Streamlit frontend for the AG News topic classifier."""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import streamlit as st
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = Path("./saved_bert_model")
MAX_LENGTH = 128

# Assignment mapping (1-based) plus 0-based HF indices used at inference
ASSIGNMENT_LABEL_MAP: Dict[int, str] = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech",
}
HF_ID_TO_LABEL: Dict[int, str] = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}


@st.cache_resource(show_spinner="Loading BERT model…")
def load_classifier() -> Tuple[AutoTokenizer, AutoModelForSequenceClassification, torch.device]:
    """Load tokenizer and model once per Streamlit session."""
    if not MODEL_DIR.exists():
        raise FileNotFoundError(
            f"Saved model not found at {MODEL_DIR}. Run training cells first."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(device)
    model.eval()
    return tokenizer, model, device


def predict_topic(text: str) -> Tuple[str, float, Dict[str, float]]:
    """Return top label, confidence (%), and per-class probabilities."""
    tokenizer, model, device = load_classifier()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100.0
    label = HF_ID_TO_LABEL.get(pred_id, "Unknown")

    class_probs = {
        HF_ID_TO_LABEL[i]: float(probs[i].item()) * 100.0 for i in range(len(probs))
    }
    return label, confidence, class_probs


def main() -> None:
    st.set_page_config(
        page_title="AG News Topic Classifier",
        page_icon="📰",
        layout="centered",
    )

    st.title("📰 AG News Topic Classifier")
    st.caption("Fine-tuned BERT (`bert-base-uncased`) — DevelopersHub Corporation")

    st.markdown(
        "Enter a news headline or article snippet below. The model predicts one of: "
        "**World**, **Sports**, **Business**, or **Sci/Tech**."
    )

    with st.expander("Label reference (assignment mapping)"):
        for class_id, name in ASSIGNMENT_LABEL_MAP.items():
            st.write(f"{class_id} → {name}")

    user_text = st.text_area(
        "News text",
        height=160,
        placeholder="e.g. Apple unveils new AI chip for next-generation MacBooks…",
    )

    classify_clicked = st.button("Classify", type="primary", use_container_width=True)

    if classify_clicked:
        if not user_text.strip():
            st.warning("Please enter some text before classifying.")
            return

        try:
            topic, confidence, class_probs = predict_topic(user_text.strip())
        except FileNotFoundError as exc:
            st.error(str(exc))
            return
        except Exception as exc:
            st.error(f"Prediction failed: {exc}")
            return

        st.success("Classification complete")

        col1, col2 = st.columns(2)
        with col1:
            st.metric(label="Predicted Topic", value=topic)
        with col2:
            st.metric(label="Confidence", value=f"{confidence:.2f}%")

        st.subheader("All class probabilities")
        for label_name, prob in sorted(class_probs.items(), key=lambda x: -x[1]):
            st.progress(min(prob / 100.0, 1.0), text=f"{label_name}: {prob:.2f}%")


if __name__ == "__main__":
    main()


Writing app.py


In [17]:
import os
os.environ["NGROK_AUTHTOKEN"] = "3EB2pnPl7BbNOB85NTyIe96L1Bt_4pNN8kYdkSjaBMUWtWP9u"  # paste token

## Cell 6: Background Streamlit & Tunnel Deployment

Launch Streamlit on port **8501** in the background, then expose it publicly with **pyngrok** (installed in Cell 1).  
Alternatively, uncomment the **localtunnel** block if you prefer `npx localtunnel`.

> **Note:** Colab may prompt you to allow ngrok; follow the link if authentication is required.

In [23]:
import os, re, subprocess, sys, time
from pathlib import Path

PORT = 8501
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "streamlit"], check=False)
subprocess.run(["pkill", "-f", "streamlit"], check=False)
time.sleep(2)

env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.address", "0.0.0.0",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--server.fileWatcherType", "none"],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(8)

proc = subprocess.Popen(
    ["npx", "--yes", "localtunnel", "--port", str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for _ in range(90):
    line = proc.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", line)
        if m:
            print("\n>>> OPEN NEW URL:", m.group(0))
            break

your url is: https://sharp-actors-carry.loca.lt

>>> OPEN NEW URL: https://sharp-actors-carry.loca.lt


### Optional: Stop background services

Run this cell when you are finished testing to free the GPU and close tunnels.

In [24]:
# Cleanup — ngrok tunnel and Streamlit process
try:
    from pyngrok import ngrok

    ngrok.kill()
    print("Ngrok tunnels closed.")
except Exception as exc:
    print(f"Ngrok cleanup skipped: {exc}")

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
print("Streamlit stopped.")

Ngrok tunnels closed.
Streamlit stopped.
